<h1>Этот ноутбук демонстрирует методы по обогащению базового датасета</h1>

<h2>Импорты библиотек и константы</h2>

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from clickhouse_driver import Client
from pyspark.sql.types import StructType, StructField, DateType, StringType, DecimalType, BooleanType
import random
from datetime import date
import glob, shutil

In [2]:
CURRENCIES     = [
    "USD", 
    "EUR", 
    "RUB", 
    "GBP"
]

CATEGORIES     = [
    "retail", 
    "food", 
    "transport", 
    "utilities", 
    "entertainment"
]

COUNTRY_CODES  = [
    "US", 
    "DE", 
    "RU", 
    "GB", 
    "FR",
    "JP"
]

CURRENCY_WEIGHTS = {
    "RUB": 0.70,
    "USD": 0.17,
    "EUR": 0.10,
    "GBP": 0.03,
}

SERVICE_LEVEL_WEIGHTS = {
    "standard": 0.70,
    "premium": 0.25,
    "vip": 0.05,
}

currencies = list(CURRENCY_WEIGHTS.keys())
weights = list(CURRENCY_WEIGHTS.values())

service_levels = list(SERVICE_LEVEL_WEIGHTS.keys())
service_weights = list(SERVICE_LEVEL_WEIGHTS.values())

In [3]:
spark = (
    SparkSession.builder
    .appName("enrichment")
    .master("local[*]")
    .config("spark.driver.memory", "2g")
    .config("spark.driver.cores", "2")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.instances", "2")
    .config("spark.executor.cores", "2")
    .config("spark.sql.shuffle.partitions", "10")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/01 22:34:05 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark: 3.5.0


<h2>Основной код</h2>

In [4]:
df_raw = (
    spark
    .read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv("./Financial_Transactions.csv")
)

df_raw.show()
df_raw.count()

+-------------+---------+-------------------+---------------+-----------------+--------------+
|TransactionID|AccountID|          Timestamp|TransactionType|TransactionAmount|AccountBalance|
+-------------+---------+-------------------+---------------+-----------------+--------------+
|        16633|     5817|2016-01-01 03:47:23|       transfer|          2446.41|      96273.47|
|        23660|     9291|2016-01-01 04:20:25|       transfer|          2640.83|      98629.95|
|        11806|     9618|2016-01-01 05:12:44|     withdrawal|           574.82|      65602.63|
|        27498|     2288|2016-01-01 05:48:42|        payment|          1740.12|      81461.66|
|         9345|     2688|2016-01-01 06:26:04|       transfer|           292.43|      18084.81|
|         5541|     3347|2016-01-01 06:57:03|     withdrawal|           534.45|       45834.3|
|        13037|     3732|2016-01-01 08:59:13|       transfer|          2687.14|      14687.11|
|        30161|     4117|2016-01-01 09:11:41|     

37417

In [5]:
rand_currency_weighted = F.udf(
    lambda: random.choices(currencies, weights=weights, k=1)[0],
    StringType()
)

rand_level_weighted = F.udf(
    lambda: random.choices(service_levels, weights=service_weights, k=1)[0],
    StringType()
)

rand_category  = F.udf(
    lambda: random.choice(CATEGORIES), 
    StringType()
)

rand_country   = F.udf(
    lambda: random.choice(COUNTRY_CODES), 
    StringType()
)

In [6]:
df_account_currency = (
    df_raw
    .select("AccountID")
    .distinct()
    .withColumn("currency", rand_currency_weighted())
    .withColumnRenamed("AccountID", "acc_id") 
)
print(f"Уникальных пользователей: {df_account_currency.count()}\n")

print("Распределение:")
df_account_currency.groupBy("currency") \
    .count() \
    .withColumn("pct", F.round(F.col("count") / df_account_currency.count() * 100, 1)) \
    .orderBy("count", ascending=False) \
    .show()

Уникальных пользователей: 8856

Распределение:
+--------+-----+----+
|currency|count| pct|
+--------+-----+----+
|     RUB| 6253|70.6|
|     USD| 1467|16.6|
|     EUR|  854| 9.6|
|     GBP|  282| 3.2|
+--------+-----+----+



In [7]:
df_enriched = (
    df_raw
    .withColumnRenamed("TransactionID", "transaction_id")
    .withColumnRenamed("AccountID", "account_id")
    .withColumnRenamed("Timestamp", "timestamp")
    .withColumnRenamed("TransactionType", "transaction_type")
    .withColumnRenamed("TransactionAmount", "amount")
    .withColumnRenamed("AccountBalance", "account_balance")
    .withColumn("transaction_type", F.lower(F.col("transaction_type")))
    .withColumn("transaction_date", F.to_date(F.col("timestamp")))
    .withColumn("merchant_category", rand_category())
    .withColumn("country_code", rand_country())
    .withColumn("account_level", rand_level_weighted())
    .join(
        df_account_currency,
        on=F.col("account_id") == F.col("acc_id"),
        how="left"
    )
    .drop("acc_id")
)

rand_category_udf = F.udf(lambda: random.choice(CATEGORIES), StringType())

df_enriched = (
    df_enriched
    .withColumn("_rand_category", rand_category_udf())
    .withColumn("merchant_category",
        F.when(F.col("transaction_type") == "payment",    F.col("_rand_category"))
         .when(F.col("transaction_type") == "withdrawal", F.lit("ATM"))
         .when(F.col("transaction_type") == "transfer",   F.lit("personal_transfer"))
         .otherwise(F.lit("other"))
    )
    .drop("_rand_category")
)

df_enriched.show()
df_enriched.count()

+--------------+----------+-------------------+----------------+-------+---------------+----------------+-----------------+------------+-------------+--------+
|transaction_id|account_id|          timestamp|transaction_type| amount|account_balance|transaction_date|merchant_category|country_code|account_level|currency|
+--------------+----------+-------------------+----------------+-------+---------------+----------------+-----------------+------------+-------------+--------+
|         16633|      5817|2016-01-01 03:47:23|        transfer|2446.41|       96273.47|      2016-01-01|personal_transfer|          FR|      premium|     RUB|
|         23660|      9291|2016-01-01 04:20:25|        transfer|2640.83|       98629.95|      2016-01-01|personal_transfer|          FR|     standard|     EUR|
|         11806|      9618|2016-01-01 05:12:44|      withdrawal| 574.82|       65602.63|      2016-01-01|              ATM|          JP|     standard|     RUB|
|         27498|      2288|2016-01-01 05

37417

In [8]:
(
    df_enriched
    .coalesce(1)
    .write
    .option("header", "true")
    .mode("overwrite")
    .csv("Financial_Transactions_Enriched")
)

files = glob.glob("Financial_Transactions_Enriched/part-*.csv")
if files:
    shutil.copy(files[0], "Financial_Transactions_Enriched.csv")
    print("Сохранено")
    
shutil.rmtree("Financial_Transactions_Enriched")

spark.stop()

Сохранено
